# L3: Pipeline de vendas 

## Initial Imports

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

In [2]:
import os
from helper import get_openai_api_key
from helper import get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## Load API tokens for our 3rd party APIs

In [3]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

## Loading Tasks and Agents YAML files

In [4]:
# Define file paths for YAML configurations
files = {
    'lead_agents': 'config/lead_qualification_agents.yaml',
    'lead_tasks': 'config/lead_qualification_tasks.yaml',
    'email_agents': 'config/email_engagement_agents.yaml',
    'email_tasks': 'config/email_engagement_tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
lead_agents_config = configs['lead_agents']
lead_tasks_config = configs['lead_tasks']
email_agents_config = configs['email_agents']
email_tasks_config = configs['email_tasks']

## Create Pydantic Models for Structured Output

In [5]:
from pydantic import BaseModel, Field
from typing import Optional, List

class LeadPersonalInfo(BaseModel):
    name: str = Field(..., description="O nome completo do lead.")
    job_title: str = Field(..., description="O cargo do lead.")
    role_relevance: int = Field(..., ge=0, le=10, description="Uma pontuação que representa quão relevante é o papel do lead no processo de tomada de decisão (0-10).")
    professional_background: Optional[str] = Field(..., description="Uma breve descrição da experiência profissional do lead.")

class CompanyInfo(BaseModel):
    company_name: str = Field(..., description="O nome da empresa onde o lead trabalha.")
    industry: str = Field(..., description="A indústria na qual a empresa atua.")
    company_size: int = Field(..., description="O tamanho da empresa em termos de número de funcionários.")
    revenue: Optional[float] = Field(None, description="A receita anual da empresa, se disponível.")
    market_presence: int = Field(..., ge=0, le=10, description="Uma pontuação que representa a presença de mercado da empresa (0-10).")

class LeadScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="A pontuação final atribuída ao lead (0-100).")
    scoring_criteria: List[str] = Field(..., description="Os critérios utilizados para determinar a pontuação do lead.")
    validation_notes: Optional[str] = Field(None, description="Quaisquer observações relacionadas à validação da pontuação do lead.")

class LeadScoringResult(BaseModel):
    personal_info: LeadPersonalInfo = Field(..., description="Informações pessoais sobre o lead.")
    company_info: CompanyInfo = Field(..., description="Informações sobre a empresa do lead.")
    lead_score: LeadScore = Field(..., description="A pontuação calculada e informações relacionadas ao lead.")


## Importing Tools

In [6]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

## Lead Qualification Crew, Agents and Tasks

In [7]:
# type: ignore
# Creating Agents
lead_data_agent = Agent(
  config=lead_agents_config['lead_data_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

cultural_fit_agent = Agent(
  config=lead_agents_config['cultural_fit_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

scoring_validation_agent = Agent(
  config=lead_agents_config['scoring_validation_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

# Creating Tasks
lead_data_task = Task(
  config=lead_tasks_config['lead_data_collection'],
  agent=lead_data_agent
)

cultural_fit_task = Task(
  config=lead_tasks_config['cultural_fit_analysis'],
  agent=cultural_fit_agent
)

scoring_validation_task = Task(
  config=lead_tasks_config['lead_scoring_and_validation'],
  agent=scoring_validation_agent,
  context=[lead_data_task, cultural_fit_task],
  output_pydantic=LeadScoringResult
)

# Creating Crew
lead_scoring_crew = Crew(
  agents=[
    lead_data_agent,
    cultural_fit_agent,
    scoring_validation_agent
  ],
  tasks=[
    lead_data_task,
    cultural_fit_task,
    scoring_validation_task
  ],
  verbose=True
)

LLM value is None
LLM value is None
LLM value is None


## Email Engagement Crew

In [8]:
# type: ignore
# Creating Agents
email_content_specialist = Agent(
  config=email_agents_config['email_content_specialist']
)

engagement_strategist = Agent(
  config=email_agents_config['engagement_strategist']
)

# Creating Tasks
email_drafting = Task(
  config=email_tasks_config['email_drafting'],
  agent=email_content_specialist
)

engagement_optimization = Task(
  config=email_tasks_config['engagement_optimization'],
  agent=engagement_strategist
)

# Creating Crew
email_writing_crew = Crew(
  agents=[
    email_content_specialist,
    engagement_strategist
  ],
  tasks=[
    email_drafting,
    engagement_optimization
  ],
  verbose=True
)

2025-01-28 14:12:05,552 - 18888 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


LLM value is None
LLM value is None


## Creating Complete Sales Flow

In [9]:
from crewai import Flow
from crewai.flow.flow import listen, start

class SalesPipeline(Flow):
    @start()
    def fetch_leads(self):
        # Puxar nossos leads do banco de dados
        leads = [
            {
                "lead_data": {
                    "name": "João Moura",
                    "job_title": "Diretor de Engenharia",
                    "company": "Clearbit",
                    "email": "joao@clearbit.com",
                    "use_case": "Usando o Agente de IA para melhorar o enriquecimento de dados."
                },
            },
        ]
        return leads

    @listen(fetch_leads)
    def score_leads(self, leads):
        scores = lead_scoring_crew.kickoff_for_each(leads)
        self.state["score_crews_results"] = scores
        return scores

    @listen(score_leads)
    def store_leads_score(self, scores):
        # Aqui armazenaríamos as pontuações no banco de dados
        return scores

    @listen(score_leads)
    def filter_leads(self, scores):
        return [score for score in scores if score['lead_score'].score > 70]

    @listen(filter_leads)
    def write_email(self, leads):
        scored_leads = [lead.to_dict() for lead in leads]
        emails = email_writing_crew.kickoff_for_each(scored_leads)
        return emails

    @listen(write_email)
    def send_email(self, emails):
        # Aqui enviaríamos os emails para os leads
        return emails

flow = SalesPipeline()

## Plotting the Flow

In [10]:
flow.plot()

Plot saved as crewai_flow.html


In [11]:
import webbrowser
webbrowser.open('crewai_flow.html')

True

## Flow Kickoff

In [12]:
emails = await flow.kickoff_async()

2025-01-28 14:12:05,763 - 18888 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


LLM value is already an LLM object
LLM value is already an LLM object
LLM value is already an LLM object
# Agent: Especialista em Dados de Leads
## Task: Coletar e analisar as seguintes informaÃ§Ãµes sobre o lead:
- InformaÃ§Ãµes Pessoais:
  - Nome: Obtenha o nome completo do lead.
  - Cargo: Determine o cargo atual do lead.
  - RelevÃ¢ncia do Cargo: Avalie quÃ£o relevante Ã© o cargo do lead para o processo de tomada de decisÃµes em uma escala de 0 a 10.
  - HistÃ³rico Profissional: Opcionalmente, colete uma breve descriÃ§Ã£o do histÃ³rico profissional do lead.

- InformaÃ§Ãµes da Empresa:
  - Nome da Empresa: Identifique o nome da empresa para a qual o lead trabalha.
  - IndÃºstria: Determine o setor em que a empresa opera.
  - Tamanho da Empresa: Estime o tamanho da empresa em termos de nÃºmero de funcionÃ¡rios.
  - Receita: Se disponÃ­vel, colete informaÃ§Ãµes sobre a receita anual da empresa.
  - PresenÃ§a no Mercado: Avalie a presenÃ§a da empresa no mercado em uma escala de 0 a 10

2025-01-28 14:12:58,273 - 18888 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed




# Agent: Avaliador e Validador de Leads
## Final Answer: 
{
  "personal_info": {
    "name": "João Moura",
    "job_title": "Diretor de Engenharia",
    "role_relevance": 9,
    "professional_background": null
  },
  "company_info": {
    "company_name": "Clearbit",
    "industry": "Business/Productivity Software",
    "company_size": 116,
    "revenue": 20000000.0,
    "market_presence": 8
  },
  "lead_score": {
    "score": 90,
    "scoring_criteria": [
      "Relevância do Cargo: 9",
      "Tamanho da Empresa: 6",
      "Presença no Mercado: 8",
      "Adequação Cultural: 9"
    ],
    "validation_notes": "A pontuação foi validada e é consistente com os dados coletados."
  }
}


LLM value is already an LLM object
LLM value is already an LLM object
# Agent: Redator de ConteÃºdo para E-mail
## Task: Redija um e-mail altamente personalizado usando o nome do lead, tÃ­tulo de trabalho, informaÃ§Ãµes da empresa e qualquer conquista pessoal ou da empresa relevante. O e-mail deve falar di

## Usage Metrics and Costs

* Vamos ver quanto custaria cada vez se essa equipe trabalhasse em grande escala.

In [13]:
import pandas as pd

# Converter a instância de UsageMetrics para um DataFrame
df_usage_metrics = pd.DataFrame([flow.state["score_crews_results"][0].token_usage.dict()])

# Calcular o custo total
custos = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Custo total: R${custos:.4f}")

# Exibir o DataFrame
df_usage_metrics

Custo total: R$0.0055


C:\Users\Pichau\AppData\Local\Temp\ipykernel_12932\3042297224.py:4: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  df_usage_metrics = pd.DataFrame([flow.state["score_crews_results"][0].token_usage.dict()])


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,36625,34492,18688,2133,10


In [14]:
import pandas as pd

# Converter a instância de UsageMetrics para um DataFrame
df_usage_metrics = pd.DataFrame([emails[0].token_usage.dict()])

# Calcular o custo total
custos = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Custo total: R${custos:.4f}")

# Exibir o DataFrame
df_usage_metrics


Custo total: R$0.0002


C:\Users\Pichau\AppData\Local\Temp\ipykernel_12932\3431595339.py:4: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  df_usage_metrics = pd.DataFrame([emails[0].token_usage.dict()])


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,1493,1236,0,257,2


## Inspecting Results

In [15]:
scores = flow.state["score_crews_results"]

In [16]:
import pandas as pd
from IPython.display import display, HTML

lead_scoring_result = scores[0].pydantic

# Criar um dicionário com a estrutura aninhada achatada
data = {
    'Nome': lead_scoring_result.personal_info.name,
    'Cargo': lead_scoring_result.personal_info.job_title,
    'Relevância do Cargo': lead_scoring_result.personal_info.role_relevance,
    'Histórico Profissional': lead_scoring_result.personal_info.professional_background,
    'Nome da Empresa': lead_scoring_result.company_info.company_name,
    'Setor': lead_scoring_result.company_info.industry,
    'Tamanho da Empresa': lead_scoring_result.company_info.company_size,
    'Receita': lead_scoring_result.company_info.revenue,
    'Presença no Mercado': lead_scoring_result.company_info.market_presence,
    'Pontuação do Lead': lead_scoring_result.lead_score.score,
    'Critérios de Pontuação': ', '.join(lead_scoring_result.lead_score.scoring_criteria),
    'Notas de Validação': lead_scoring_result.lead_score.validation_notes
}

# Converter o dicionário para um DataFrame
df = pd.DataFrame.from_dict(data, orient='index', columns=['Valor'])

# Resetar o índice para transformar os nomes originais das colunas em uma coluna comum
df = df.reset_index()

# Renomear a coluna do índice para 'Atributo'
df = df.rename(columns={'index': 'Atributo'})

# Criar tabela HTML com atributos em negrito e valores alinhados à esquerda
html_table = df.style.set_properties(**{'text-align': 'left'}) \
                     .format({'Atributo': lambda x: f'<b>{x}</b>'}) \
                     .hide(axis='index') \
                     .to_html()

# Exibir a tabela HTML estilizada
display(HTML(html_table))

Atributo,Valor
Nome,João Moura
Cargo,Diretor de Engenharia
Relevância do Cargo,9
Histórico Profissional,None
Nome da Empresa,Clearbit
Setor,Business/Productivity Software
Tamanho da Empresa,116
Receita,20000000.000000
Presença no Mercado,8
Pontuação do Lead,90


## Results

In [17]:
import textwrap

result_text = emails[0].raw
wrapped_text = textwrap.fill(result_text, width=80)
print(wrapped_text)

João,   Aumento de produtividade e eficiência é crucial para a Clearbit. Com a
plataforma CrewAI, você pode orquestrar Agentes de IA para automação de
processos que potencialmente transformarão suas operações.   **Pronto para ver
como isso pode funcionar especificamente para sua equipe?** Clique [aqui](#)
para agendar uma reunião rápida e vamos personalizar a solução ideal para suas
necessidades.   Se preferir, me diga um horário que funcione para você e farei a
organização.   **Não perca essa oportunidade de impulsionar seu crescimento!**


## Quão complexo isso pode se tornar?

In [20]:
from crewai import Flow
from crewai.flow.flow import listen, start, and_, or_, router

class SalesPipeline(Flow):
    
  @start()
  def fetch_leads(self):
    # Buscar leads do banco de dados
    # Isto é um mock. Em um cenário real, aqui você buscaria
    # leads de um banco de dados.
    leads = [
      {
        "lead_data": {
          "name": "João Moura",
          "job_title": "Diretor de Engenharia",
          "company": "Clearbit",
          "email": "joao@clearbit.com",
          "use_case": "Usando Agente de IA para melhorar o enriquecimento de dados."
        },
      },
    ]
    return leads

  @listen(fetch_leads)
  def score_leads(self, leads):
    scores = lead_scoring_crew.kickoff_for_each(leads)
    self.state["score_crews_results"] = scores
    return scores

  @listen(score_leads)
  def store_leads_score(self, scores):
    # Aqui armazenaríamos as pontuações no banco de dados
    return scores

  @listen(score_leads)
  def filter_leads(self, scores):
    return [score for score in scores if score['lead_score'].score > 70]

  @listen(and_(filter_leads, store_leads_score))
  def log_leads(self, leads):
    print(f"Leads: {leads}")

  @router(filter_leads)
  def count_leads(self, scores):
    if len(scores) > 10:
      return 'alto'
    elif len(scores) > 5:
      return 'médio'
    else:
      return 'baixo'

  @listen('alto')
  def store_in_salesforce(self, leads):
    return leads

  @listen('médio')
  def send_to_sales_team(self, leads):
    return leads

  @listen('baixo')
  def write_email(self, leads):
    scored_leads = [lead.to_dict() for lead in leads]
    emails = email_writing_crew.kickoff_for_each(scored_leads)
    return emails

  @listen(write_email)
  def send_email(self, emails):
    # Aqui enviaríamos os e-mails para os leads
    return emails


## Plotting the Flow

In [21]:
flow = SalesPipeline()
flow.plot()

Plot saved as crewai_flow.html


In [28]:
import webbrowser
webbrowser.open('crewai_flow.html')

True